# PBMC 3k single-cell RNA-seq: a scverse best-practices walkthrough

**Goal.** Go from a raw 10x Genomics count matrix to annotated immune cell types, explaining each decision.

**Dataset.** 2,700 peripheral blood mononuclear cells (PBMCs) from a healthy donor, 10x Chromium v1, `hg19`.

**Pipeline.** load → QC → MAD filtering → normalise/log1p → HVG → PCA → kNN graph → Leiden → UMAP → marker-based annotation.

The reusable version of every step lives in [`src/pbmc3k_pipeline.py`](../src/pbmc3k_pipeline.py); this notebook imports its helpers and narrates the reasoning.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

import pbmc3k_pipeline as pp

SEED = 0
ROOT = Path.cwd().parent
FIGDIR = ROOT / "figures"
FIGDIR.mkdir(exist_ok=True)
sc.set_figure_params(dpi=100, frameon=False, figsize=(5, 5))
sc.settings.verbosity = 1

def save(fig, name):
    pp.savefig(fig, FIGDIR, name)

## 1. Load the data
`read_10x_mtx` reads the sparse `matrix.mtx` plus gene and barcode tables into an `AnnData` object (cells × genes). Gene symbols are not guaranteed unique in the reference, so we make them unique.

In [2]:
adata = sc.read_10x_mtx(ROOT / "data/filtered_gene_bc_matrices/hg19", var_names="gene_symbols")
adata.var_names_make_unique()
adata

AnnData object with n_obs × n_vars = 2700 × 32738
    var: 'gene_ids'
    layers: None (.X)

## 2. Quality control
Low-quality droplets show up as (a) very few counts/genes (empty or damaged cells), (b) a high fraction of mitochondrial reads (cytoplasmic mRNA leaked out of a dying cell), or (c) unusually high counts (potential doublets). We compute per-cell metrics, including ribosomal and haemoglobin fractions.

In [3]:
pp.add_qc_metrics(adata)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, m in zip(axes, ["n_genes_by_counts", "total_counts", "pct_counts_mt"]):
    sc.pl.violin(adata, m, ax=ax, show=False)
save(fig, "01_qc_violins_prefilter")
adata.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].describe().round(2)

,n_genes_by_counts,total_counts,pct_counts_mt
count,2700.00,2700.00,2700.00
mean,846.99,2366.90,2.22
std,282.10,1094.26,1.17
min,212.00,548.00,0.00
25%,690.00,1757.75,1.54
50%,817.00,2197.00,2.03
75%,953.25,2763.00,2.64
max,3422.00,15844.00,22.57


### MAD-based filtering
Fixed cutoffs (e.g. "mito < 5%") do not transfer between tissues or protocols. Instead we flag cells that deviate by more than **5 median absolute deviations (MADs)** from the dataset median on log-total-counts, log-genes and %counts in the top 20 genes, and on %mito, with an extra hard ceiling at 8% mito. 5 MADs is deliberately permissive: the scverse best-practices book recommends erring on the side of keeping cells and revisiting if downstream clusters look noisy.

*Why not 3 MADs for mito?* In this dataset the mito MAD is only ~0.5%, so 3 MADs ≈ 3.6% and would discard ~7–8% of cells that look healthy. 5 MADs (≈4.7%) is close to the classic Seurat 5% rule.

In [4]:
pp.flag_low_quality(adata, nmads=5, mt_nmads=5, mt_floor=8)
keep = ~(adata.obs["outlier"] | adata.obs["mt_outlier"])
print(f"Removing {(~keep).sum()} of {adata.n_obs} cells "
      f"({adata.obs['outlier'].sum()} count/gene outliers, {adata.obs['mt_outlier'].sum()} mito outliers)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt", ax=axes[0], show=False)
sc.pl.scatter(adata, "total_counts", "pct_counts_mt", color="outlier", ax=axes[1], show=False)
save(fig, "02_qc_scatter_prefilter")

Removing 159 of 2700 cells (101 count/gene outliers, 76 mito outliers)


In [5]:
adata = adata[keep].copy()
sc.pp.filter_genes(adata, min_cells=3)     # drop genes seen in <3 cells
print(adata)

AnnData object with n_obs × n_vars = 2541 × 13542
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier'
    var: 'gene_ids', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
    layers: None (.X)


> **Not done here:** ambient-RNA correction (SoupX/CellBender) and doublet detection (Scrublet/scDblFinder). The 10x v1 chemistry and low loading make doublets rare in this dataset, but both are standard in production pipelines.

## 3. Normalise and log-transform
Cells differ in sequencing depth for technical reasons. We scale each cell to 10,000 counts (`normalize_total`) and apply `log1p` to stabilise variance. Raw counts are preserved in `layers["counts"]`, and the log-normalised matrix is stored in `.raw` for plotting and differential expression.

In [6]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata

## 4. Highly variable genes
Most genes carry little biological signal across cells. Selecting the 2,000 most variable genes focuses PCA on informative structure and reduces noise.

In [7]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat")
sc.pl.highly_variable_genes(adata, show=False)
save(plt.gcf(), "04_highly_variable_genes")
print(adata.var["highly_variable"].sum(), "HVGs")

2000 HVGs


## 5. PCA
We scale the HVG subset (clip at ±10 to prevent a few extreme cells dominating) and run PCA. The elbow in the variance-ratio plot guides how many PCs to carry forward; here 30 sits safely past the elbow.

In [8]:
hv = adata[:, adata.var["highly_variable"]].copy()
sc.pp.scale(hv, max_value=10)
sc.tl.pca(hv, n_comps=50, svd_solver="arpack", random_state=SEED)
adata.obsm["X_pca"] = hv.obsm["X_pca"]
adata.uns["pca"] = hv.uns["pca"]
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True, show=False)
save(plt.gcf(), "05_pca_variance_ratio")

/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/functools.py:912: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


## 6. Neighbour graph, Leiden clustering, UMAP
- **kNN graph** (k=15, 30 PCs): cells are nodes, edges join transcriptionally similar cells.
- **Leiden** community detection finds densely connected groups. Resolution controls granularity; 1.0 resolves T-cell subsets that 0.5 merges.
- **UMAP** is for *visualisation only*; clusters are defined on the graph, not on the 2-D embedding.

In [9]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, random_state=SEED)
sc.tl.leiden(adata, resolution=1.0, flavor="leidenalg", n_iterations=-1,
             directed=False, random_state=SEED, key_added="leiden")
sc.tl.umap(adata, random_state=SEED)
ax = sc.pl.umap(adata, color="leiden", legend_loc="on data", show=False, title="Leiden (resolution=1.0)")
save(ax.figure, "06_umap_leiden")
adata.obs["leiden"].value_counts().sort_index()

/var/folders/mf/c51djqk94kzfdclzk__ctxsc0000gn/T/ipykernel_63615/1753351889.py:2: UserWarning: The `igraph` implementation of leiden clustering is *orders of magnitude faster*. Set the flavor argument to (and install if needed) 'igraph' to use it.
  sc.tl.leiden(adata, resolution=1.0, flavor="leidenalg", n_iterations=-1,


leiden
0    608
1    526
2    406
3    329
4    301
5    204
6    139
7     28
Name: count, dtype: int64

## 7. Marker genes and annotation
First, an unbiased look: Wilcoxon rank-sum test for genes enriched in each cluster. Then we compare against canonical PBMC markers.

In [10]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon", use_raw=True)
pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(6)

,0,1,2,3,4,5,6,7
0,RPS12,IL32,S100A9,CD74,CCL5,LST1,NKG7,HLA-DPB1
1,RPS27,LTB,LYZ,CD79A,NKG7,COTL1,GZMB,FCER1A
2,RPL32,LDHB,S100A8,HLA-DRA,CST7,FCER1G,PRF1,HLA-DPA1
3,RPS6,CD3D,TYROBP,CD79B,B2M,AIF1,GNLY,CST3
4,RPS14,IL7R,FCN1,HLA-DPB1,GZMA,FTH1,CTSW,HLA-DRA
5,RPS25,TPT1,CST3,HLA-DQA1,CTSW,IFITM3,FGFBP2,HLA-DRB1


Reading the table against textbook biology:

| Cluster signature | Genes | Cell type |
|---|---|---|
| `IL7R`, `CCR7`, `LEF1`, `TCF7` | naive T | Naive CD4 T |
| `IL7R`, `S100A4` | memory/effector T | Memory CD4 T |
| `CD8A`, `CD8B`, `CCL5`, `GZMK` | cytotoxic T | CD8 T |
| `GNLY`, `NKG7`, `GZMB`, `PRF1` | cytolytic, no CD3 | NK |
| `MS4A1`, `CD79A` | B-cell receptor complex | B |
| `CD14`, `LYZ`, `S100A8/9` | classical monocyte | CD14+ Mono |
| `FCGR3A`, `MS4A7`, `COTL1` | non-classical monocyte | FCGR3A+ Mono |
| `FCER1A`, `CLEC10A` | cDC | Dendritic |

To make labelling reproducible rather than eyeballed, we score every cell for each marker set (`score_genes`), average per cluster, and assign the highest-scoring type. The score table is kept so any call can be audited.

In [11]:
table = pp.annotate_clusters(adata)
mapping = table.idxmax(axis=1)
adata.obs["cell_type"] = adata.obs["leiden"].map(mapping.to_dict()).astype("category")
table.round(2).assign(assigned=mapping)

,Naive CD4 T,Memory CD4 T,CD8 T,NK,B,CD14+ Mono,FCGR3A+ Mono,Dendritic,Megakaryocyte,assigned
leiden,,,,,,,,,,
0,0.10,-0.19,-0.40,-0.69,-0.37,-0.46,-0.27,-0.52,-0.02,Naive CD4 T
1,-0.08,0.40,-0.54,-0.71,-0.36,-0.51,-0.19,-0.52,-0.02,Memory CD4 T
2,-0.77,-0.05,-0.71,-0.71,0.01,2.99,0.93,0.02,0.01,CD14+ Mono
3,-0.58,-1.03,-0.61,-0.65,2.37,-0.43,-0.25,0.56,-0.01,B
4,-0.46,0.28,0.98,0.74,-0.27,-0.58,-0.21,-0.33,-0.00,CD8 T
5,-0.88,-0.16,-0.80,-0.77,0.14,0.86,2.01,0.11,0.12,FCGR3A+ Mono
6,-0.86,-0.21,-0.09,3.02,-0.32,-0.63,0.05,-0.49,-0.04,NK
7,-0.87,-0.28,-0.83,-0.80,0.30,1.24,0.64,1.98,-0.00,Dendritic


In [12]:
flat = list(dict.fromkeys(g for v in pp.MARKERS.values() for g in v if g in adata.raw.var_names))
dp = sc.pl.dotplot(adata, flat, groupby="leiden", standard_scale="var", show=False, return_fig=True)
dp.savefig(FIGDIR / "10_marker_dotplot_by_cluster.png", dpi=150, bbox_inches="tight")
dp = sc.pl.dotplot(adata, flat, groupby="cell_type", standard_scale="var", show=False, return_fig=True)
dp.savefig(FIGDIR / "13_marker_dotplot_by_celltype.png", dpi=150, bbox_inches="tight")
plt.show()

/var/folders/mf/c51djqk94kzfdclzk__ctxsc0000gn/T/ipykernel_63615/97934944.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
ax = sc.pl.umap(adata, color="cell_type", show=False, title="PBMC 3k - annotated")
save(ax.figure, "11_umap_cell_types")
canon = ["IL7R", "CCR7", "CD8A", "NKG7", "GNLY", "MS4A1", "CD14", "LYZ", "FCGR3A", "FCER1A", "PPBP"]
axs = sc.pl.umap(adata, color=canon, ncols=4, show=False, use_raw=True)
save(axs[0].figure, "12_umap_canonical_markers")

In [14]:
counts = adata.obs["cell_type"].value_counts()
pd.DataFrame({"cells": counts, "percent": (counts / counts.sum() * 100).round(1)})

,cells,percent
cell_type,,
Naive CD4 T,608,23.9
Memory CD4 T,526,20.7
CD14+ Mono,406,16.0
B,329,12.9
CD8 T,301,11.8
FCGR3A+ Mono,204,8.0
NK,139,5.5
Dendritic,28,1.1


## 8. Sanity checks and caveats
- **Biology agrees with expectation**: ~56% T cells (CD4 + CD8), ~24% monocytes, ~13% B, ~5% NK, ~1% DC — typical for healthy PBMC.
- **Naive vs memory CD4** split on a continuum (see UMAP): the boundary is soft, so treat as a gradient, not a hard class.
- **Megakaryocytes/platelets** (`PPBP`) are present in the original Seurat tutorial (~14 cells) but fall below cluster resolution here and are not a separate cluster.
- **Dendritic cells** are only ~28 cells; the label is supported by `FCER1A`/`CLEC10A`, but small clusters deserve orthogonal validation (e.g. CITE-seq).
- Labels are manual-marker based. For larger studies consider CellTypist or reference mapping (scANVI) to scale and cross-check.

## 9. Why this matters for immunology and drug discovery
Bulk blood profiling averages across cell types, so a drug that depletes B cells or expands CD8 T cells looks like a vague shift in expression. Single-cell resolution lets you ask *which* cell population responds. Applied the same way, this workflow supports:
- **Target discovery**: finding genes specific to a pathogenic subset (e.g. an interferon-high monocyte) rather than the whole tissue.
- **Pharmacodynamics**: tracking cell-type composition and activation state before and after treatment.
- **Patient stratification**: using immune-cell state as a biomarker for responders vs non-responders.

Getting QC, normalisation and annotation right is the foundation: every downstream claim (differential expression, trajectories, cell-cell communication) inherits whatever errors are introduced here.